# Agentic Workflow: Parallelization Pattern

## 1. Setup and Client Initialization

We load the environment variables from the `.env` file in the parent directory and initialize the OpenAI client pointing to the Gemini API's OpenAI-compatible endpoint.

In [8]:
import os
import json
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env in the parent directory
dir_path = os.path.dirname(__file__) if "__file__" in globals() else "."
dotenv_path = os.path.abspath(os.path.join(dir_path, "..", ".env"))
load_dotenv(dotenv_path=dotenv_path)

# Retrieve the API key from environment variables
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_actual_api_key_here":
    raise ValueError(
        "Please set your GEMINI_API_KEY in the .env file in the root of the project.\n"
        "You can get a free key from Google AI Studio: https://aistudio.google.com/"
    )

# Initialize the OpenAI client pointing to Gemini's OpenAI-compatible API endpoint
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# We use the free gemini-3.1-flash-lite model via the OpenAI compatible endpoint.
model_name = "gemini-3.1-flash-lite"
print("Gemini API client initialized successfully!")

Gemini API client initialized successfully!


## 2. Local Directory Initialization

We load source files directly from the `code_to_review` folder. This simulates reviewing local source files representing code changes.

In [11]:
db_dir = os.path.join(dir_path, "code_to_review")

def load_local_pr():
    if not os.path.exists(db_dir):
        raise FileNotFoundError(f"Source directory not found at {db_dir}")
    
    files = []
    for filename in os.listdir(db_dir):
        if filename.endswith(".py"):
            file_path = os.path.join(db_dir, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            files.append({
                "filename": filename,
                "content": content
            })
            
    pr_details = {
        "pr_id": "PR-LOCAL",
        "title": "Review of local files in code_to_review directory",
        "description": "Consolidated code review of Python source files found in the code_to_review directory.",
        "author": "local_dev",
        "files": files
    }
    return pr_details

# Verify directory load
pr_details = load_local_pr()
print(f"Loaded PR with {len(pr_details['files'])} files from '{db_dir}' to review.")

Loaded PR with 2 files from '.\code_to_review' to review.


## 3. Base Completion Function

A helper function to get chat completions from the Gemini model using specific instructions.

In [12]:
def get_completion(prompt, system_instruction="You are a helpful customer support assistant."):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

## 4. Auditor Agent Prompts

We define specialized system instructions for each auditor agent, highlighting the specific focus areas they should evaluate in the codebase.

In [13]:
SECURITY_SYSTEM_INSTRUCTION = (
    "You are a Senior Security Auditor. Analyze the provided Pull Request code and "
    "identify any security vulnerabilities, exposures, or unsafe practices. "
    "Provide your findings in a clear, bulleted list. If no security issues are found, "
    "explicitly state 'No security issues found.'\n\n"
    "Key Focus Areas:\n"
    "- SQL injection vulnerabilities\n"
    "- Hardcoded credentials or API keys\n"
    "- Unsafe code execution (e.g., eval(), exec())\n"
    "- Insecure inputs handling\n"
    "- Clear explanations of potential impact and suggested remediations"
)

PERFORMANCE_SYSTEM_INSTRUCTION = (
    "You are a Senior Performance Engineer. Analyze the provided Pull Request code and "
    "identify any execution bottlenecks, algorithmic inefficiencies, or resource leaks. "
    "Provide your findings in a clear, bulleted list. If no performance issues are found, "
    "explicitly state 'No performance issues found.'\n\n"
    "Key Focus Areas:\n"
    "- O(N^2) or worse algorithmic complexities on collections\n"
    "- N+1 database query patterns (database hits inside loops)\n"
    "- Unnecessary memory consumption\n"
    "- Redundant resource connections or missing cleanup (e.g. database cursors, file streams)\n"
    "- Concrete suggestions for optimization"
)

STYLE_SYSTEM_INSTRUCTION = (
    "You are a Lead Software Architect. Analyze the provided Pull Request code for "
    "style guidelines, readability, naming conventions, docstrings, and maintainability. "
    "Provide your findings in a clear, bulleted list. If no style/maintainability issues "
    "are found, explicitly state 'No style/maintainability issues found.'\n\n"
    "Key Focus Areas:\n"
    "- Adherence to standard Python conventions (like PEP 8)\n"
    "- Variable, function, and class naming clarity\n"
    "- Missing docstrings, documentation, or type hints\n"
    "- Code structure and modularity (overly long functions)"
)

## 5. Synthesizer Agent (PR Commenter) Prompt

Agent D acts as the synthesizer. It takes the output reports from the three parallel agents and compiles them into a unified, clean, and developer-friendly Markdown comment.

In [14]:
SYNTHESIZER_SYSTEM_INSTRUCTION = (
    "You are a Principal Software Engineer acting as a PR Commenter. Your job is to aggregate, "
    "deduplicate, and cleanly summarize the reports from three specialized auditors (Security, "
    "Performance, Style & Maintainability) into a single, cohesive Markdown review comment.\n\n"
    "Guidelines:\n"
    "- Maintain a constructive and professional developer tone.\n"
    "- Remove any redundant comments or overlapping feedback.\n"
    "- Structure your final response using the following headers:\n"
    "  ### 📑 PR Review Summary\n"
    "  (A 2-3 sentence overview of the overall code quality and critical concerns)\n\n"
    "  ### 🔒 Security Audit\n"
    "  (Summarize security findings. Use a sub-bullet list for vulnerabilities and remediations. If clean, write 'No security issues found.')\n\n"
    "  ### ⚡ Performance & Optimization\n"
    "  (Summarize performance improvements. If clean, write 'No performance issues found.')\n\n"
    "  ### 🛠️ Style, Docs & Maintainability\n"
    "  (Summarize style guidelines violations. If clean, write 'No style/maintainability issues found.')\n\n"
    "  ### 🚦 Final Decision\n"
    "  (Choose one of: **APPROVE**, **REQUEST CHANGES**. Justify the decision based on severity of findings. If there are security vulnerabilities or severe performance issues, you MUST select 'REQUEST CHANGES'.)"
)

## 6. Parallel Execution Implementation

We write the orchestrator function. It launches tasks in parallel using a `ThreadPoolExecutor`, waits for all audits to finish, compiles the reports, and passes them to the Synthesizer.

In [15]:
def run_agent_review(agent_name, system_instruction, pr_details):
    print(f"[{agent_name}] Analysis started...")
    # Construct the review prompt with the files code and meta
    prompt = (
        f"PR Title: {pr_details['title']}\n"
        f"PR Description: {pr_details['description']}\n\n"
        f"Files to review:\n"
    )
    for f in pr_details["files"]:
        prompt += f"=== Filename: {f['filename']} ===\n{f['content']}\n\n"
    
    result = get_completion(prompt, system_instruction)
    print(f"[{agent_name}] Analysis complete.")
    return result

def review_pull_request(pr_details):
    print(f"\n{'='*20} Initiating PR Review for {pr_details['pr_id']} {'='*20}")
    print(f"Title: {pr_details['title']}")
    print(f"Author: {pr_details['author']}\n")
    
    agents = [
        ("Security Auditor", SECURITY_SYSTEM_INSTRUCTION),
        ("Performance Optimizer", PERFORMANCE_SYSTEM_INSTRUCTION),
        ("Style & Maintainability", STYLE_SYSTEM_INSTRUCTION)
    ]
    
    reviews = {}
    # Execute the three audits in parallel to minimize latency
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(run_agent_review, name, instr, pr_details): name
            for name, instr in agents
        }
        for future in futures:
            name = futures[future]
            try:
                reviews[name] = future.result()
            except Exception as e:
                print(f"Error running {name}: {e}")
                reviews[name] = f"Failed to run review: {e}"
                
    print("\n[Synthesizer] Review aggregation started...")
    
    # Compile individual reviews for synthesis
    synth_prompt = (
        f"PR Title: {pr_details['title']}\n"
        f"PR Author: {pr_details['author']}\n\n"
        f"=== SECURITY AUDITOR REPORT ===\n{reviews['Security Auditor']}\n\n"
        f"=== PERFORMANCE OPTIMIZER REPORT ===\n{reviews['Performance Optimizer']}\n\n"
        f"=== STYLE & MAINTAINABILITY REPORT ===\n{reviews['Style & Maintainability']}\n"
    )
    
    final_comment = get_completion(synth_prompt, SYNTHESIZER_SYSTEM_INSTRUCTION)
    print("[Synthesizer] Review aggregation complete.\n")
    
    return reviews, final_comment

## 7. Run Review Workflows

We load the pull request files from the local directory, run the parallel review workflow, and print the synthesized results.

In [7]:
pr_details = load_local_pr()
individual_reviews, final_pr_comment = review_pull_request(pr_details)

print(f"\n{'#'*80}")
print(f"### FINAL SYNTHESIZED REVIEW FOR {pr_details['pr_id']} ###")
print(f"{'#'*80}\n")
print(final_pr_comment)
print("\n" + "*"*80 + "\n")


==================== Initiating PR Review for PR-LOCAL ====================
Title: Review of local files in code_to_review directory
Author: local_dev

[Security Auditor] Analysis started...
[Performance Optimizer] Analysis started...
[Style & Maintainability] Analysis started...
[Security Auditor] Analysis complete.
[Style & Maintainability] Analysis complete.
[Performance Optimizer] Analysis complete.

[Synthesizer] Review aggregation started...
[Synthesizer] Review aggregation complete.


################################################################################
### FINAL SYNTHESIZED REVIEW FOR PR-LOCAL ###
################################################################################

### 📑 PR Review Summary
The current submission contains critical security vulnerabilities, including Remote Code Execution, SQL Injection, and hardcoded credentials, which must be addressed before any further review. Additionally, the codebase suffers from significant performance bottlenecks 